## What is key
A key is just a sequence of random bytes. Think of it as an extremely long password. The longer and more random it is, the harder it is to guess.Fluree requires a 32-byte key (256 bits). That is 32 numbers, each between 0 and 255.

In [5]:
import secrets 

# Generate a 32 cryptographically secure random bytes
key_bytes = secrets.token_bytes(32)

print(f"Type: {type(key_bytes)}")
print(f"{key_bytes}")
print(f"Length: {len(key_bytes)} bytes = {len(key_bytes) * 8} bits")
print(f"As hex: {key_bytes.hex()}")
print(f"As ints : {list(key_bytes[:8])} ...")  # first 8 numbers

Type: <class 'bytes'>
b'\x95\xa0\x85-\x1b\xb5{\x06c\x01.\xcd\xaeg\x11\xd0\xee\x85\xeb\n%;\xb7\xeb\xce"J\x81\xf4\r\x95\xcd'
Length: 32 bytes = 256 bits
As hex: 95a0852d1bb57b0663012ecdae6711d0ee85eb0a253bb7ebce224a81f40d95cd
As ints : [149, 160, 133, 45, 27, 181, 123, 6] ...


In [4]:
import secrets
import base64

key_bytes = secrets.token_bytes(32)

# Encode to base64 -> safe to put in config file or env variable

key_b64 = base64.b64encode(key_bytes).decode()

print(f"Raw bytes length : {len(key_bytes)}")
print(f"Base64 string    : {key_b64}")
print(f"Base64 length    : {len(key_b64)} characters")

# Decode back to bytes: always get 32 bytes
decoded = base64.b64decode(key_b64)
print(f"Decoded length   : {len(decoded)} bytes")
print(f"Round-trip OK    : {key_bytes == decoded}")


Raw bytes length : 32
Base64 string    : j0OY3hVfIHTyPn1nzQYrZtsvZq/pyQQlHR59d0+TsXI=
Base64 length    : 44 characters
Decoded length   : 32 bytes
Round-trip OK    : True


Note: A valid Fluree encryption key is always exactly 32 bytes, stored as 44-character Base64 string

## AES-256-GCM — the algorithm Fluree uses

In [8]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import secrets

# My 32-byte key
key = secrets.token_bytes(32)

# GCM needs a nonce (number used once)- 12 bytes, random every time
nonce = secrets.token_bytes(12)

# Create an AEGGCM cipher with my key
cipher = AESGCM(key)

# Encrypt some data
plaintext  = b"Hello, secret world!"
ciphertext = cipher.encrypt(nonce, plaintext, associated_data=None)

print(f"Plaintext  ({len(plaintext):2d} bytes): {plaintext}")
print(f"Ciphertext ({len(ciphertext):2d} bytes): {ciphertext.hex()}")
print(f"Overhead: {len(ciphertext) - len(plaintext)} bytes (that's the 16-byte GCM tag)")

# Decrypt with correct key and nonce -> get back to plaintext
recovered = cipher.decrypt(nonce, ciphertext, associated_data=None)
print(f"Decrypted              : {recovered}")
print(f"Match                  : {recovered == plaintext}")

Plaintext  (20 bytes): b'Hello, secret world!'
Ciphertext (36 bytes): f6a3bc4949b6a296910c01a73f62940b0ee5be6267219fead08219475043ce06f5ee1b25
Overhead: 16 bytes (that's the 16-byte GCM tag)
Decrypted              : b'Hello, secret world!'
Match                  : True


## What happens with a WRONG key?

In [11]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag
import secrets

key_correct=secrets.token_bytes(32)
key_wrong=secrets.token_bytes(32) # Completely different key

nonce=secrets.token_bytes(12)

cipher_correct=AESGCM(key_correct)
cipher_wrong=AESGCM(key_wrong)

ciphertext = cipher_correct.encrypt(nonce, b"Top secret data", None)

try:
    cipher_wrong.decrypt(nonce, ciphertext, None)
    print("Decrypted successfully- THIS SHOULD NOT HAPPEN!")

except InvalidTag:
    print("Invalid tag- the data was not decrypted")
    print("GCM authentication tag did not match → decryption aborted")
    print("No data was returned. This is the correct behaviour.")



Invalid tag- the data was not decrypted
GCM authentication tag did not match → decryption aborted
No data was returned. This is the correct behaviour.


Key insight: With GCM, a wrong key does not return garbage data, it returns nothing at all and raises an error. You cannot even accidentally read scrambled data

## How Fluree stores encrypted data

Every file fluree writes starts with 22-bytes header followed by the encrypted data

```
Byte offset   Content               Size      Meaning
─────────────────────────────────────────────────────────────
0–3           Magic bytes           4 bytes   Always "FLU\0" (0x46 0x4C 0x55 0x00)
4             Version               1 byte    Format version (0x01)
5             Algorithm             1 byte    0x01 = AES-256-GCM
6–9           Key ID                4 bytes   Which key was used (for key rotation)
10–21         Nonce                 12 bytes  Random, unique per write
─────────────────────────────────────────────────────────────
22+           Ciphertext            variable  Your actual encrypted data
last 16       Authentication tag    16 bytes  GCM integrity checksum
```

The magic bytes `FLU\0` at the start are your **quick check** — if the first
4 bytes of a Fluree file are `46 4c 55 00`, encryption is on.


In [12]:
import struct
import secrets
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

MAGIC     = b"FLU\x00"
VERSION   = b"\x01"
ALGORITHM = b"\x01"         # 0x01 = AES-256-GCM
KEY_ID    = struct.pack(">I", 1) 

print(KEY_ID)

b'\x00\x00\x00\x01'


In [13]:
key = secrets.token_bytes(32)
nonce = secrets.token_bytes(12)

header = MAGIC + VERSION + ALGORITHM + KEY_ID + nonce
print(f"Header ({len(header)} bytes): {header.hex()}")
print(f"  Magic    : {MAGIC.hex()} → {MAGIC}")
print(f"  Version  : {VERSION.hex()}")
print(f"  Alg      : {ALGORITHM.hex()}")
print(f"  Key ID   : {KEY_ID.hex()}")
print(f"  Nonce    : {nonce.hex()}")

Header (22 bytes): 464c5500010100000001f6ed2ae4b2d6cf77a899f2ca
  Magic    : 464c5500 → b'FLU\x00'
  Version  : 01
  Alg      : 01
  Key ID   : 00000001
  Nonce    : f6ed2ae4b2d6cf77a899f2ca


In [14]:
# Encrypt the payload
cipher = AESGCM(key)
plaintext = b'{"@id": "ex:Alice", "ex:secret": "TOP_SECRET_VALUE_12345"}'

# AAD = Additional Authenticated Data: the header is included in the tag

ciphertext= cipher.encrypt(nonce, plaintext, associated_data=header)

envelope = header + ciphertext
print(f"\nFull envelope ({len(envelope)} bytes):")
print(f"  First 4 bytes : {envelope[:4].hex()} ← magic bytes FLU\\0")
print(f"  Total         : {len(envelope)} bytes")
print(f"  Plaintext was : {len(plaintext)} bytes")
print(f"  Overhead      : {len(envelope) - len(plaintext)} bytes (22 header + 16 tag)")


Full envelope (96 bytes):
  First 4 bytes : 464c5500 ← magic bytes FLU\0
  Total         : 96 bytes
  Plaintext was : 58 bytes
  Overhead      : 38 bytes (22 header + 16 tag)


## The encryption key: generating and validating
### Generate Key

In [17]:
import secrets
import base64

def generate_fluree_key()-> str:
    """Generate a cryptographically secure AES-256 key for Fluree."""
    key_bytes = secrets.token_bytes(32) # 32 bytes = 256 bits
    return base64.b64encode(key_bytes).decode() # base64 encoded for easy storage



In [18]:
key = generate_fluree_key()
print(f"Your Fluree encryption key:")
print(f"  {key}")
print(f"\nSave this somewhere safe. If you lose it, you lose your data.")
print(f"Never commit it to Git.")

Your Fluree encryption key:
  8o81ucUMY5dzewA5OL+lcoYq4N2ycnu9JeB7DSQTUTk=

Save this somewhere safe. If you lose it, you lose your data.
Never commit it to Git.


### Validate Key before using it

In [ ]:
import base64
import binascii


def validate_fluree_key(b64_key:str)-> None:
    """ Validate that a string is a proper Fluree AES-256 key. Raises ValueError with a clear message if not.
    Args:
        b64_key (str): The base64 encoded key to validate
    Raises:
        ValueError: If the key is not a valid Fluree AES-256 key
    """

    # Must not empty
    if not b64_key:
        raise ValueError("Key must not be empty")
    
    # Must be a valid Base64 string
    try:
        key_bytes = base64.b64decode(b64_key)
    except (binascii.Error, ValueError) as exc:
        raise ValueError(f"Not valid Base64: {exc}") from exc
    
    # Must be 32 bytes
    if len(key_bytes) != 32:
        raise ValueError(
            f"Key must be 32 bytes (AES-256). "
            f"Got {len(key_bytes)} bytes after Base64 decoding. "
            f"Common mistake: using the raw string instead of base64-encoding it."
        )
    
    print(f"Key is valid: {len(key_bytes)} bytes = {len(key_bytes) * 8} bits")
        

### Test with a valid key


In [21]:
import secrets
good_key = base64.b64encode(secrets.token_bytes(32)).decode()
validate_fluree_key(good_key)



Key is valid: 32 bytes = 256 bits


### Test with the placeholder from the docker-compose file


In [22]:
print()
try:
    validate_fluree_key("FLUREE_ENCRYPTION_KEY")   # the mistake in docker-compose
except ValueError as exc:
    print(f"Error: {exc}")




Error: Not valid Base64: Incorrect padding


### Test with an empty string

In [23]:
print()

try:
    validate_fluree_key("")
except ValueError as exc:
    print(f"Error: {exc}")


Error: Key must not be empty


## Connect Fluree to our key

The key reaches Fluree through an environment variable called FLUREE_ENCRYPTION_KEY. This is set before the server starts and the server reads it once at startup.

### Step1: In Docker (local development)

In [ ]:
import secrets
import base64
import os

def write_env_file(path:str=".env")-> str:
    """Generate a key and write it to a .env file in the current directory."""
    key = generate_fluree_key()

    with open(path, "w") as f:
        f.write(f"FLUREE_ENCRYPTION_KEY={key}\n")
    
    print(f"Written to {path}")
    print(f"Key (first 8 chars): {key[:8]}...")
    print(f"NEVER commit this file to Git.")
    return key

# Run once
key = write_env_file()



Written to .env
Key (first 8 chars): JqZo1PwM...
NEVER commit this file to Git.


### Step2: Then run: docker-compose up -d
docker-compose.yml reads: FLUREE_ENCRYPTION_KEY: "${FLUREE_ENCRYPTION_KEY}"

## Verify the server is using the key

In [25]:
import requests
def check_fluree_health(url:str="http://localhost:8090")-> bool:
    """Check if Fluree is running and ready to use."""
    try:
        r = requests.get(f"{url}/health", timeout=5)
        print(f"Status Code: {r.status_code}")
        print(f"Response: {r.text}")
        return r.status_code == 200
    
    except requests.ConnectionError:
        print("Cannot reach Fluree. Is docker-compose up running?")
        return False
    
    except Exception as e:
        print(f"Unexpected error: {e}")
        return False
    

check_fluree_health()

Status Code: 200
Response: {"status":"ok","version":"4.0.3"}


True

## Writing and reading data: Implement the Full lifecycle

Here is the full journey of a piece of sensitive data through Fluree, step by step

### Step1: Create Ledger

A ledger in Fluree is like a database. We must create it before writing data. The format is name:branch

In [26]:
import requests

FLUREE_URL = "http://localhost:8090"
LEDGER = "demo:main"

def create_ledger(ledger:str)-> None:
    """Create a new ledger with the given name and branch."""
    r = requests.post(f"{FLUREE_URL}/v1/fluree/create", json={"ledger": ledger})
    if r.status_code in (200,201):
        print(f"Ledger {ledger} created successfully")
    elif r.status_code == 409:
        print(f"Ledger {ledger} already exists")
    else:
        raise RuntimeError(f"Create failed: {r.status_code} {r.text}")

In [27]:
create_ledger(LEDGER)

Ledger demo:main created successfully
